In [2]:

# ============================================================
# 1. Model and Pricing
# ============================================================

MODEL = "gemini-3.5-flash"

# Price for 1 million tokens (USD - Standard Paid Tier)
INPUT_PRICE = 0.30
OUTPUT_PRICE = 2.50

# ============================================================
# 2. Load Question Dataset
# ============================================================

import pandas as pd

df = pd.read_csv("ai_teacher_20_questions.csv")

print(f"Total Questions: {len(df)}")

df = df.head(5).copy()

print(f"Questions selected for API: {len(df)}")

# ============================================================
# 3. Import Libraries and Load API Key
# ============================================================

import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Gemini API Key not found in .env file")

client = genai.Client(api_key=api_key)

# ============================================================
# 4. AI Teacher Prompt
# ============================================================

instructions = """
You are an expert AI Teacher.

Explain the concept using simple layman terms.
Keep the answer precise and easy to understand.
Give the answer in approximately 3 lines.
"""

# ============================================================
# 5. Storage
# ============================================================

answers = []
latencies = []

input_tokens = []
output_tokens = []
total_tokens = []

input_costs = []
output_costs = []
total_costs = []

# ============================================================
# 6. Generate Answers
# ============================================================

for index, row in df.iterrows():

    question = str(row["User Query"])

    try:

        # Start Time
        start_time = time.perf_counter()

        # Gemini API Call
        response = client.models.generate_content(
            model=MODEL,
            contents=question,
            config=types.GenerateContentConfig(
                system_instruction=instructions,
                temperature=0
            )
        )

        # End Timer
        end_time = time.perf_counter()

        # Latency
        latency = end_time - start_time

        # Answer
        answer = response.text.strip() if response.text else ""

        # Token Usage
        usage = response.usage_metadata

        input_token_count = usage.prompt_token_count or 0
        output_token_count = usage.candidates_token_count or 0
        total_token_count = usage.total_token_count or 0

        # Cost
        input_cost = (
            input_token_count / 1_000_000
        ) * INPUT_PRICE

        output_cost = (
            output_token_count / 1_000_000
        ) * OUTPUT_PRICE

        total_cost = input_cost + output_cost

        # Store Values
        answers.append(answer)
        latencies.append(latency)

        input_tokens.append(input_token_count)
        output_tokens.append(output_token_count)
        total_tokens.append(total_token_count)

        input_costs.append(input_cost)
        output_costs.append(output_cost)
        total_costs.append(total_cost)

        print(f"Question {index + 1} completed")

    except Exception as e:

        print(f"Question {index + 1} Error: {e}")

        answers.append("ERROR")
        latencies.append(0)

        input_tokens.append(0)
        output_tokens.append(0)
        total_tokens.append(0)

        input_costs.append(0)
        output_costs.append(0)
        total_costs.append(0)

# ============================================================
# 7. Add Results to DataFrame
# ============================================================

df["GeneratedAnswer"] = answers
df["Latency"] = latencies

df["InputTokens"] = input_tokens
df["OutputTokens"] = output_tokens
df["TotalTokens"] = total_tokens

df["InputCosts"] = input_costs
df["OutputCosts"] = output_costs
df["TotalCosts"] = total_costs

# ============================================================
# 8. Save Results
# ============================================================

df.to_csv(
    "ai_teacher_evaluation_operational_metrics.csv",
    index=False
)

# ============================================================
# 9. Display DataFrame
# ============================================================

display(
    df[
        [
            "User Query",
            "GeneratedAnswer",
            "Latency",
            "InputTokens",
            "OutputTokens",
            "TotalTokens",
            "InputCosts",
            "OutputCosts",
            "TotalCosts"
        ]
    ]
)


Total Questions: 20
Questions selected for API: 5
Question 1 completed
Question 2 Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 17h20m53.054592574s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate

,User Query,GeneratedAnswer,Latency,InputTokens,OutputTokens,TotalTokens,InputCosts,OutputCosts,TotalCosts
0,What is AI?,Artificial Intelligence (AI) is technology tha...,4.311472,44,78,531,0.000013,0.000195,0.000208
1,How does AI differ from traditional programming?,ERROR,0.000000,0,0,0,0.000000,0.000000,0.000000
2,What is Machine Learning?,ERROR,0.000000,0,0,0,0.000000,0.000000,0.000000
3,What is Deep Learning?,ERROR,0.000000,0,0,0,0.000000,0.000000,0.000000
4,"What is the difference between AI, ML, and Dee...",ERROR,0.000000,0,0,0,0.000000,0.000000,0.000000


In [7]:
# Load the New Result:
df = pd.read_csv("ai_teacher_evaluation_operational_metrics.csv")

# convert metric column to numeric:

metric_columns = [
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCosts",
    "OutputCosts",
    "TotalCosts"
]

for column in metric_columns:

    df[column] = pd.to_numeric(df[column], errors = "coerce")

# Average Metric:

average_latency = df["Latency"].mean()
average_input_tokens = df["InputTokens"].mean()
average_output_tokens = df["OutputTokens"].mean()
average_total_tokens = df["TotalTokens"].mean()
average_input_costs = df["InputCosts"].mean()
average_output_costs = df["OutputCosts"].mean()
average_total_costs = df["TotalCosts"].mean()

# Total Metrics

total_latency = df["Latency"].sum()
total_input_tokens = df["InputTokens"].sum()
total_output_tokens = df["OutputTokens"].sum()
total_tokens = df["TotalTokens"].sum()
total_input_costs = df["InputCosts"].sum()
total_output_costs = df["OutputCosts"].sum()
total_costs = df["TotalCosts"].sum()

# Create Operational Metrics DataFrame

operational_metrics_df = pd.DataFrame({
    "Metrics": [
        "Latency (seconds)",
        "Input Tokens",
        "Output Tokens",
        "Total Tokens",
        "Input Costs(USD)",
        "Output Costs(USD)",
        "Total Costs(USD)"
    ],

    "Average":[
        average_latency,
        average_input_tokens,
        average_output_tokens,
        average_total_tokens,
        average_input_costs,
        average_output_costs,
        average_total_costs
    ],

    "Total": [
        total_latency,
        total_input_tokens,
        total_output_tokens,
        total_tokens,
        total_input_costs,
        total_output_costs,
        total_costs
    ]
})

# Round Values:

operational_metrics_df["Average"] = (operational_metrics_df["Average"].round(6))
operational_metrics_df["Total"] = (operational_metrics_df["Total"].round(6))

# Display DataFrame:

operational_metrics_df

,Metrics,Average,Total
0,Latency (seconds),0.862294,4.311472
1,Input Tokens,8.800000,44.000000
2,Output Tokens,15.600000,78.000000
3,Total Tokens,106.200000,531.000000
4,Input Costs(USD),0.000003,0.000013
5,Output Costs(USD),0.000039,0.000195
6,Total Costs(USD),0.000042,0.000208


In [8]:
operational_metrics_df.to_csv(
    "ai_teacher_evaluation_operational_metrics_average.csv",
    index = False,
    encoding = "utf-8-sig"
)